# DSCNet (arXiv:2405.04913) — ResNet-101 + Optuna WSSS (4 Phương thức) trên Pascal VOC 2012

Notebook này triển khai paper **[\[2405.04913\] Weakly-supervised Semantic Segmentation via Dual-stream Contrastive Learning of Cross-image Contextual Information](https://arxiv.org/abs/2405.04913)** (**DSCNet**) trên backbone **`resnet101`** (`layer4 stride=1`, `448×448` → feature map `28×28`) và đánh giá theo khung **Optuna WSSS** cho **4 phương thức**:

1. **`Vanilla CAM`** (Refined CAM $\hat{O}_i$ từ DSCL)
2. **`CAM + SAM`**
3. **`LayerCAM`** (`layer2 + layer3 + layer4`)
4. **`LayerCAM + SAM`**

### Kiến trúc & Cơ chế chính của DSCNet
- **PGCL (Pixel-wise Group Contrastive Learning, Eq. 4–7)**: Tính tương quan Pearson giữa các pixel, phân nhóm rời rạc theo số lớp $|Y_i| + 1$ bằng K-means và học tương phản InfoNCE giữa các nhóm pixel xuyên ảnh.
- **SGCL (Semantic-wise Graph Contrastive Learning, Eq. 8–9)**: Xây dựng đồ thị ngữ nghĩa GCN $(K+1)$ lớp kết hợp ma trận đồng xuất hiện nhãn mức ảnh và độ tương đồng nhóm–lớp $M_{i,u}^k$, tối ưu tương phản ngữ nghĩa xuyên ảnh.
- **DSCL Feature Refinement (Eq. 10–13)**: Tổng hợp đặc trưng ngữ cảnh $\tilde{f}_i = \text{softmax}(f_i \otimes \mathcal{C}_i^T) \otimes \mathcal{S}_i$, ghép nối $\hat{f}_i = [f_i, \tilde{f}_i]$ để sinh Refined CAM $\hat{O}_i$ với hàm loss $\mathcal{L}_{DSCL} = \alpha \mathcal{L}_{PGCL} + \beta \mathcal{L}_{SGCL} + \mathcal{L}_{CE}$ ($\alpha = 0.6$, $\beta = 0.4$).

### Chia tập dữ liệu Pascal VOC 2012 (`0%` rò rỉ dữ liệu)
- **Tập huấn luyện classifier**: `Main/trainval \ Seg/val` (**10,418 ảnh** — đã loại bỏ toàn bộ `1,449` ảnh `Segmentation/val.txt`).
- **Tập tune Optuna**: **310 ảnh** cân bằng lớp từ `Segmentation/train.txt \ train_ids` (`0%` trùng train & val).
- **Tập đánh giá cuối**: `Segmentation/val.txt` (**1,449 ảnh**, đánh giá ở độ phân giải gốc, tái sử dụng SAM image embeddings).

## 1. Cấu hình thí nghiệm (`ResNet-101`, `DSCNet`, `Optuna 310 ảnh`, `4 phương thức`)

- `SAM_MODEL_TYPE = "auto"`: tự chọn `"vit_l"` khi chạy trên Colab (chuẩn đồng nhất với `Optuna_WSSS_Pipeline.ipynb`), hoặc ưu tiên checkpoint có sẵn khi chạy local (`vit_l` nếu có sẵn, nếu không dùng `vit_b` tại `checkpoints/sam_vit_b_01ec64.pth`). Có thể ép trực tiếp `"vit_b"`, `"vit_l"`, hoặc `"vit_h"`.
- `TUNE_SUBSET_SIZE = 310`: dùng trọn bộ **310 ảnh** cân bằng (từ `Seg/train \ train_ids`) để Optuna tối ưu siêu tham số cho cả 4 phương pháp.

In [ ]:
# =====================================================================
# 1. CẤU HÌNH TỔNG THỂ — DSCNet (arXiv:2405.04913) + RESNET-101
# =====================================================================
MODEL_NAME = "resnet101"
IMAGE_SIZE = 448
LAYER4_STRIDE_1 = True            # layer4[0] stride=(1, 1) -> feature map 28x28 @448

TRAIN_SPLIT = "main_trainval"     # Main/trainval \ Seg/val -> 10,418 ảnh
DSCNET_MAX_TRAIN_IMAGES = None    # Đặt int (vd 1000) để test nhanh; None = đủ 10,418 ảnh

BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 15
LR_SCHEDULER = "poly"
POLY_POWER = 0.9
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP = True
SEED = 42
NUM_WORKERS = 2                   # Tự chuyển thành 0 trên Windows local nếu cần
REQUIRE_GPU = True

# --- Tham số DSCNet (arXiv:2405.04913) ---
DSCNET_EMBED_DIM = 256
DSCNET_ALPHA_PGCL = 0.6           # Mục IV-E của paper: alpha tối ưu = 0.6
DSCNET_BETA_SGCL = 0.4            # Mục IV-E của paper: beta tối ưu = 0.4
DSCNET_TEMPERATURE = 0.2          # Nhiệt độ InfoNCE cho PGCL & SGCL
DSCNET_GRAPH_TEMP = 0.2           # Nhiệt độ tương đồng nhóm -> node đồ thị SGCL
DSCNET_KMEANS_ITERS = 6           # Số vòng lặp K-means theo tương quan Pearson
DSCNET_POS_CORR_THR = 0.5         # Ngưỡng tương quan dương trong nhóm (Eq. 5)
DSCNET_WARMUP_EPOCHS = 1          # Epoch 1 ramp nhẹ tương phản để ổn định CAM khởi tạo

# --- CAM / LayerCAM lúc Tune & Eval ---
CAM_SCALES = (0.75, 1.0, 1.25)
CAM_HFLIP = True
LAYERCAM_LAYERS = ("layer2", "layer3", "layer4")
LAYERCAM_SHALLOW_GAMMA = 2.0
CLS_THRESHOLD = 0.5
USE_NEG_POINTS = True

# --- Optuna (Tập tune đúng 310 ảnh) ---
CAM_TUNE_MAX_IMAGES = 310
TUNE_SUBSET_SIZE = 310
OPTUNA_CLS_TRIALS = 51
OPTUNA_CAM_TRIALS = 35
OPTUNA_SAM_TRIALS = 60
OPTUNA_LAYERCAM_TRIALS = 35
OPTUNA_LAYERCAM_SAM_TRIALS = 60
TUNE_PROTOCOL_VERSION = 1

# --- SAM ("auto" | "vit_b" | "vit_l" | "vit_h") ---
SAM_MODEL_TYPE = "auto"

# --- Quản lý Run / Checkpoint / Đối chứng Baseline ---
EXPERIMENT_NAME = "DSCNet_ResNet101_LayerCAM"
BASELINE_RUN_NAME = "PaperWSSS_3Backbones_MainTrainVal_448_Stride1"
COMPARE_WITH_BASELINE = True      # Tự nạp baseline resnet101 nếu đã có trên Drive/local
TRAIN_BASELINE_IF_MISSING = False # Đặt True nếu muốn train thêm baseline ResNet-101 từ đầu
RESUME_IF_EXISTS = True
RETRAIN_DSCNET = False

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS331_WSSS"
REPO_URL = "https://github.com/minhthanh224/CS331-Project.git"
REPO_BRANCH = "dscnet-resnet101-layercam"

METHODS = ["Vanilla CAM", "CAM + SAM", "LayerCAM", "LayerCAM + SAM"]

assert MODEL_NAME == "resnet101"
assert TRAIN_SPLIT == "main_trainval"
assert SAM_MODEL_TYPE in {"auto", "vit_b", "vit_l", "vit_h"}
assert DSCNET_ALPHA_PGCL >= 0 and DSCNET_BETA_SGCL >= 0
assert 0 < TUNE_SUBSET_SIZE <= CAM_TUNE_MAX_IMAGES
assert all(l in {"layer1", "layer2", "layer3", "layer4"} for l in LAYERCAM_LAYERS)

## 2. Thiết lập môi trường Colab / Local, cài đặt thư viện & nạp `src`

In [ ]:
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    missing_pkgs = []
    for mod_name, pkg_spec in [
        ("optuna", "optuna"),
        ("sklearn", "scikit-learn"),
        ("segment_anything", "git+https://github.com/facebookresearch/segment-anything.git"),
    ]:
        if importlib.util.find_spec(mod_name) is None:
            missing_pkgs.append(pkg_spec)
    if missing_pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)

    DRIVE_BASE = Path(DRIVE_PROJECT_DIR)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

    PROJECT_ROOT = Path("/content/CS331-Project")
    if not (PROJECT_ROOT / ".git").exists():
        if PROJECT_ROOT.exists():
            shutil.rmtree(PROJECT_ROOT)
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "origin", REPO_BRANCH], check=False)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name in ("notebooks", "test"):
        PROJECT_ROOT = PROJECT_ROOT.parent
    DRIVE_BASE = PROJECT_ROOT
    if os.name == "nt":
        NUM_WORKERS = 0  # An toàn cho DataLoader trong Jupyter trên Windows

project_root = PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torchvision.datasets.utils import check_integrity, download_url, extract_archive
from tqdm.auto import tqdm

from src import (
    SAM_MODEL_SPECS,
    VOC_CLASSES as CLASSES,
    DSCNetClassifier,
    SegmentationEvaluator,
    VOCClassificationDataset,
    build_balanced_tune_table,
    build_classifier,
    build_dscnet_optimizer,
    build_dscnet_optuna_pipeline,
    build_optimizer,
    build_optuna_pipeline,
    build_sam_predictor,
    build_transforms,
    build_voc_cooccurrence_matrix,
    build_voc_label_table,
    cache_cams,
    cls_summary_row,
    compute_classification_metrics,
    ensure_sam_checkpoint,
    evaluate_segmentation,
    gen_cam_pseudo_mask,
    gen_sam_pseudo_mask,
    get_device,
    get_sam_embedding,
    image_cell,
    load_classifier,
    load_dscnet_classifier,
    load_eval_sample,
    load_eval_samples,
    log_eval_results,
    make_cam_fn,
    make_dscnet_cam_fn,
    make_loader,
    mask_cell,
    plot_grid,
    poly_scheduler,
    predict_probs,
    predict_sample_masks,
    read_split_ids,
    read_wsss_train_ids,
    seg_summary_row,
    set_sam_embedding,
    set_seed,
    train_dscnet_one_epoch,
    train_one_epoch,
)

set_seed(SEED)
device = get_device(require_gpu=REQUIRE_GPU)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DRIVE_BASE   : {DRIVE_BASE}")
print(f"DEVICE       : {device} ({gpu_name})")
print(f"NUM_WORKERS  : {NUM_WORKERS}")

## 3. Chuẩn bị thư mục lưu kết quả, bộ dữ liệu Pascal VOC 2012 & checkpoint SAM

In [ ]:
drive_run_dir = DRIVE_BASE / "runs" / EXPERIMENT_NAME
run_dir = drive_run_dir if IN_COLAB else (project_root / "runs" / EXPERIMENT_NAME)
ckpt_dir = run_dir / "checkpoints"
plots_dir = run_dir / "plots"
optuna_dir = run_dir / "optuna"
for d in (run_dir, ckpt_dir, plots_dir, optuna_dir):
    d.mkdir(parents=True, exist_ok=True)
for tag in ("resnet101_dscnet", "resnet101_baseline"):
    (ckpt_dir / tag).mkdir(parents=True, exist_ok=True)

# Resolve SAM_MODEL_TYPE
if SAM_MODEL_TYPE == "auto":
    if IN_COLAB:
        SAM_MODEL_TYPE = "vit_l"
    else:
        local_ckpt_dir = project_root / "checkpoints"
        vit_l_name, _, vit_l_min = SAM_MODEL_SPECS["vit_l"]
        if (local_ckpt_dir / vit_l_name).is_file() and (local_ckpt_dir / vit_l_name).stat().st_size >= vit_l_min:
            SAM_MODEL_TYPE = "vit_l"
        else:
            SAM_MODEL_TYPE = "vit_b"

archive_name = "VOCtrainval_11-May-2012.tar"
archive_md5 = "6cd6e144f989b92b3379bac3b3de84fd"

if IN_COLAB:
    archive_dir = DRIVE_BASE / "archives"
    archive_dir.mkdir(parents=True, exist_ok=True)

    data_root = Path("/content/data")
    data_root.mkdir(parents=True, exist_ok=True)
    voc_root = data_root / "VOCdevkit" / "VOC2012"
    local_archive = data_root / archive_name
    drive_archive = archive_dir / archive_name

    if not voc_root.is_dir():
        if not check_integrity(local_archive, archive_md5):
            if drive_archive.is_file():
                print("Đang copy VOC2012 archive từ Google Drive...")
                shutil.copy2(drive_archive, local_archive)
            else:
                urls = [
                    "https://data.pjreddie.com/files/" + archive_name,
                    "https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/" + archive_name,
                ]
                for url in urls:
                    try:
                        print(f"Đang tải VOC2012 từ {url} ...")
                        download_url(url, str(data_root), filename=archive_name, md5=archive_md5)
                        break
                    except Exception as exc:
                        print(f"Tải thất bại: {url} ({exc})")
                else:
                    raise RuntimeError("Không thể tải VOC2012 tự động. Hãy upload archive lên Google Drive.")

        if not check_integrity(local_archive, archive_md5):
            raise RuntimeError("VOC2012 archive bị sai MD5!")
        extract_archive(str(local_archive), str(data_root))
        if not drive_archive.exists():
            shutil.copy2(local_archive, drive_archive)

    local_sam_ckpt = ensure_sam_checkpoint(SAM_MODEL_TYPE, data_root, cache_dirs=[archive_dir])
    drive_sam_ckpt = archive_dir / SAM_MODEL_SPECS[SAM_MODEL_TYPE][0]
    if not drive_sam_ckpt.is_file():
        shutil.copy2(local_sam_ckpt, drive_sam_ckpt)
else:
    voc_root = project_root / "data" / "VOC2012"
    local_sam_ckpt = ensure_sam_checkpoint(SAM_MODEL_TYPE, project_root / "checkpoints")

print("VOC2012        :", voc_root)
print(f"SAM ({SAM_MODEL_TYPE}) :", local_sam_ckpt)
print("Run dir        :", run_dir)

## 4. Chia tập dữ liệu VOC 2012 (`10,418` Train / `310` Optuna Tune / `1,449` Val)

- **Train (`Main/trainval \ Seg/val`)**: 10,418 ảnh, tuyệt đối không chứa bất kỳ ảnh nào thuộc `Segmentation/val.txt`.
- **Tune Optuna**: 310 ảnh cân bằng lớp lấy từ `Segmentation/train.txt \ train_ids` (`0%` trùng Train và Val).
- **Eval**: 1,449 ảnh `Segmentation/val.txt`.

In [ ]:
seg_train_ids = read_split_ids(voc_root, "Segmentation", "train")
val_ids = read_split_ids(voc_root, "Segmentation", "val")
train_ids = read_wsss_train_ids(voc_root, TRAIN_SPLIT)
assert len(train_ids) == 10418, f"Expected 10,418 train IDs, got {len(train_ids)}"
assert len(seg_train_ids) == 1464 and len(val_ids) == 1449

if DSCNET_MAX_TRAIN_IMAGES is not None:
    train_ids = train_ids[: int(DSCNET_MAX_TRAIN_IMAGES)]

assert set(train_ids).isdisjoint(val_ids)
assert set(seg_train_ids).isdisjoint(val_ids)

df_train = build_voc_label_table(voc_root, train_ids, CLASSES)
df_seg_train = build_voc_label_table(voc_root, seg_train_ids, CLASSES)
df_val = build_voc_label_table(voc_root, val_ids, CLASSES)
df_tune_balanced = build_balanced_tune_table(
    df_seg_train, train_ids, classes=CLASSES, max_images=CAM_TUNE_MAX_IMAGES, seed=SEED
)

assert set(df_tune_balanced["image_id"]).isdisjoint(val_ids)
if DSCNET_MAX_TRAIN_IMAGES is None:
    assert len(df_tune_balanced) == 310, f"Expected 310 tune images, got {len(df_tune_balanced)}"
    assert set(df_tune_balanced["image_id"]).isdisjoint(train_ids)

print(f"1. Tập Train Classifier (Main/trainval \\ Seg/val): {len(df_train):5d} ảnh (0% trùng Seg/val)")
print(f"2. Tập Seg/train đầy đủ                          : {len(df_seg_train):5d} ảnh")
print(f"   -> Tập Tune Optuna cân bằng                   : {len(df_tune_balanced):5d} ảnh")
print(f"3. Tập Eval cuối cùng (Segmentation/val.txt)     : {len(df_val):5d} ảnh")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
x = np.arange(len(CLASSES))
w = 0.28
axes[0].bar(x - w, df_train[list(CLASSES)].sum(), width=w, label=f"Train ({len(df_train):,})")
axes[0].bar(x, df_seg_train[list(CLASSES)].sum(), width=w, label=f"Seg/train ({len(df_seg_train):,})")
axes[0].bar(x + w, df_val[list(CLASSES)].sum(), width=w, label=f"Seg/val ({len(df_val):,})")
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASSES, rotation=45, ha="right")
axes[0].set_ylabel("Số lượng ảnh")
axes[0].set_title("Phân bố 20 lớp: Train / Seg-train / Val")
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.35)

axes[1].bar(x, df_tune_balanced[list(CLASSES)].sum(), color="#2b8cbe", width=0.55)
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASSES, rotation=45, ha="right")
axes[1].set_ylabel("Số lượng ảnh")
axes[1].set_title(f"Tập Tune Cân Bằng cho Optuna ({len(df_tune_balanced)} ảnh)")
axes[1].grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(plots_dir / "class_distribution_splits.png", dpi=160)
plt.show()

train_transform = build_transforms(IMAGE_SIZE, train=True)
eval_transform = build_transforms(IMAGE_SIZE, train=False)

train_dataset = VOCClassificationDataset(voc_root, df_train, CLASSES, train_transform)
seg_train_dataset = VOCClassificationDataset(voc_root, df_seg_train, CLASSES, eval_transform)
tune_balanced_dataset = VOCClassificationDataset(voc_root, df_tune_balanced, CLASSES, eval_transform)
val_dataset = VOCClassificationDataset(voc_root, df_val, CLASSES, eval_transform)

seg_train_loader = make_loader(seg_train_dataset, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device)
val_loader = make_loader(val_dataset, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device)
criterion = nn.BCEWithLogitsLoss()

## 5. Huấn luyện `DSCNet` (`ResNet-101` + `PGCL` + `SGCL` + `DSCL`)

- Khởi tạo ma trận đồng xuất hiện ngữ nghĩa `cooccur_adj` từ **chỉ nhãn cấp ảnh** của tập Train (`df_train`) — không dùng bất kỳ pixel mask nào.
- Tối ưu đồng thời $\mathcal{L}_{DSCL} = \alpha \mathcal{L}_{PGCL} + \beta \mathcal{L}_{SGCL} + \mathcal{L}_{CE}$ ($\alpha=0.6, \beta=0.4$) với Poly LR (`power=0.9`), AMP và lưu checkpoint tốt nhất theo `Val mAP`.
- Nếu có checkpoint Baseline `resnet101` từ các lần chạy trước (`BASELINE_RUN_NAME` hoặc `checkpoints/resnet101/`), tự động nạp để đối chứng trực tiếp.

In [ ]:
config_record = {
    "experiment_name": EXPERIMENT_NAME,
    "paper_arxiv": "2405.04913",
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "layer4_stride1": LAYER4_STRIDE_1,
    "train_split": TRAIN_SPLIT,
    "train_num_images": len(train_ids),
    "tune_num_images": len(df_tune_balanced),
    "tune_subset_size": TUNE_SUBSET_SIZE,
    "val_num_images": len(val_ids),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler": LR_SCHEDULER,
    "poly_power": POLY_POWER,
    "dscnet_embed_dim": DSCNET_EMBED_DIM,
    "dscnet_alpha_pgcl": DSCNET_ALPHA_PGCL,
    "dscnet_beta_sgcl": DSCNET_BETA_SGCL,
    "dscnet_temperature": DSCNET_TEMPERATURE,
    "dscnet_graph_temp": DSCNET_GRAPH_TEMP,
    "dscnet_kmeans_iters": DSCNET_KMEANS_ITERS,
    "dscnet_pos_corr_thr": DSCNET_POS_CORR_THR,
    "dscnet_warmup_epochs": DSCNET_WARMUP_EPOCHS,
    "cam_scales": list(CAM_SCALES),
    "cam_hflip": CAM_HFLIP,
    "layercam_layers": list(LAYERCAM_LAYERS),
    "layercam_shallow_gamma": LAYERCAM_SHALLOW_GAMMA,
    "use_neg_points": USE_NEG_POINTS,
    "sam_model": SAM_MODEL_TYPE,
    "tune_protocol_version": TUNE_PROTOCOL_VERSION,
    "eval_protocol": "original_resolution",
    "seed": SEED,
}
(run_dir / "config.json").write_text(json.dumps(config_record, indent=2, ensure_ascii=False), encoding="utf-8")

trained_models = {}
all_histories = {}

dsc_dir = ckpt_dir / "resnet101_dscnet"
dsc_best_path = dsc_dir / "best.pt"
dsc_last_path = dsc_dir / "last.pt"
dsc_weights_path = dsc_dir / "best_weights_only.pt"


def _inspect_dscnet_ckpt(path: Path):
    if not path.is_file():
        return None
    try:
        payload = torch.load(path, map_location="cpu", weights_only=True)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            return None
        hist = payload.get("history", [])
        ep = int(payload.get("epoch_in_run") or (hist[-1]["epoch"] if hist else 0))
        val_m = float(payload.get("val_mAP", hist[-1]["val_mAP"] if hist else -1.0))
        return {"path": path, "epoch": ep, "val_mAP": val_m, "history": hist, "config": payload.get("config", {})}
    except Exception:
        return None


best_meta = _inspect_dscnet_ckpt(dsc_best_path)
last_meta = _inspect_dscnet_ckpt(dsc_last_path)

if RESUME_IF_EXISTS and not RETRAIN_DSCNET and best_meta is not None:
    trained_models["resnet101_dscnet"] = load_dscnet_classifier(
        best_meta["path"], device=device, backbone_name=MODEL_NAME, num_classes=len(CLASSES),
        embed_dim=DSCNET_EMBED_DIM, layer4_stride1=LAYER4_STRIDE_1,
    )
    full_hist = last_meta["history"] if (last_meta and len(last_meta["history"]) >= len(best_meta["history"])) else best_meta["history"]
    all_histories["resnet101_dscnet"] = full_hist
    dscnet_selected_info = {
        "ckpt_name": best_meta["path"].name,
        "epoch": int(best_meta["epoch"]),
        "val_mAP": float(best_meta["val_mAP"]),
    }
    print(f"[Resume] Nạp DSCNet ResNet-101 từ {best_meta['path']} (epoch={dscnet_selected_info['epoch']}, val_mAP={dscnet_selected_info['val_mAP']:.4f})")
else:
    cooccur_adj = build_voc_cooccurrence_matrix(df_train, CLASSES)
    train_loader = make_loader(
        train_dataset, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, device=device, seed=SEED
    )
    dsc_model = DSCNetClassifier(
        backbone_name=MODEL_NAME,
        num_classes=len(CLASSES),
        embed_dim=DSCNET_EMBED_DIM,
        layer4_stride1=LAYER4_STRIDE_1,
        pretrained=True,
        kmeans_iters=DSCNET_KMEANS_ITERS,
        pos_corr_thr=DSCNET_POS_CORR_THR,
        graph_temp=DSCNET_GRAPH_TEMP,
        pred_cls_threshold=CLS_THRESHOLD,
        cooccur_adj=cooccur_adj,
    ).to(device)

    optimizer = build_dscnet_optimizer(
        dsc_model, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = poly_scheduler(
        optimizer, total_steps=NUM_EPOCHS * len(train_loader), power=POLY_POWER
    )
    scaler = torch.amp.GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None

    history = []
    best_val_map = -1.0
    best_epoch = 0

    print("=" * 88)
    print(
        f"TRAIN DSCNet RESNET-101 | {NUM_EPOCHS} epochs | {len(train_dataset):,} ảnh | "
        f"alpha_pgcl={DSCNET_ALPHA_PGCL}, beta_sgcl={DSCNET_BETA_SGCL}, temp={DSCNET_TEMPERATURE}"
    )
    print("=" * 88)

    for epoch in range(1, NUM_EPOCHS + 1):
        current_lr = optimizer.param_groups[0]["lr"]
        if DSCNET_WARMUP_EPOCHS > 0 and epoch <= DSCNET_WARMUP_EPOCHS:
            contrast_scale = float(epoch) / float(DSCNET_WARMUP_EPOCHS + 1)
        else:
            contrast_scale = 1.0

        train_stats = train_dscnet_one_epoch(
            model=dsc_model,
            loader=train_loader,
            optimizer=optimizer,
            cls_criterion=criterion,
            device=device,
            scheduler=scheduler,
            alpha_pgcl=DSCNET_ALPHA_PGCL,
            beta_sgcl=DSCNET_BETA_SGCL,
            temperature=DSCNET_TEMPERATURE,
            contrast_scale=contrast_scale,
            use_amp=USE_AMP,
            scaler=scaler,
            desc=f"[DSCNet] Ep {epoch:02d}/{NUM_EPOCHS:02d} (c_scale={contrast_scale:.2f})",
        )

        val_probs, val_targets, val_loss = predict_probs(
            dsc_model, val_loader, device, criterion=criterion,
            desc=f"[DSCNet] Ep {epoch:02d} [Val]", leave=False, use_amp=USE_AMP,
        )
        val_cls = compute_classification_metrics(val_targets, val_probs, threshold=CLS_THRESHOLD)
        val_map = val_cls["mAP"]
        val_macro_f1 = val_cls["macro_f1"]

        improved = val_map > best_val_map
        if improved:
            best_val_map = val_map
            best_epoch = epoch

        history.append({
            "backbone": "resnet101_dscnet",
            "epoch": epoch,
            "backbone_lr": current_lr,
            "contrast_scale": round(contrast_scale, 4),
            "train_loss": round(train_stats["loss"], 6),
            "ce_loss": round(train_stats["ce_loss"], 6),
            "pgcl_loss": round(train_stats["pgcl_loss"], 6),
            "sgcl_loss": round(train_stats["sgcl_loss"], 6),
            "val_loss": round(val_loss, 6),
            "val_mAP": round(val_map, 6),
            "val_macro_f1": round(val_macro_f1, 6),
        })

        payload = {
            "backbone": MODEL_NAME,
            "variant": "resnet101_dscnet",
            "epoch_in_run": epoch,
            "best_epoch": best_epoch,
            "num_classes": len(CLASSES),
            "classes": list(CLASSES),
            "model_state_dict": dsc_model.state_dict(),
            "val_loss": val_loss,
            "val_mAP": val_map,
            "best_val_mAP": best_val_map,
            "history": list(history),
            "config": config_record,
        }
        torch.save(payload, dsc_last_path)
        if improved:
            torch.save(payload, dsc_best_path)
            torch.save(dsc_model.state_dict(), dsc_weights_path)

        (dsc_dir / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
        pd.DataFrame(history).to_csv(dsc_dir / "training_history.csv", index=False)

        star = " [BEST]" if improved else ""
        print(
            f"[DSCNet] Ep {epoch:02d}/{NUM_EPOCHS:02d} | LR: {current_lr:.2e} | "
            f"Loss: {train_stats['loss']:.4f} (ce={train_stats['ce_loss']:.4f}, "
            f"pgcl={train_stats['pgcl_loss']:.4f}, sgcl={train_stats['sgcl_loss']:.4f}) | "
            f"Val mAP: {val_map:.4f} | F1: {val_macro_f1:.4f}{star}"
        )

    dscnet_selected_info = {
        "ckpt_name": "best.pt",
        "epoch": int(best_epoch),
        "val_mAP": float(best_val_map),
    }
    trained_models["resnet101_dscnet"] = load_dscnet_classifier(
        dsc_best_path, device=device, backbone_name=MODEL_NAME, num_classes=len(CLASSES),
        embed_dim=DSCNET_EMBED_DIM, layer4_stride1=LAYER4_STRIDE_1,
    )
    all_histories["resnet101_dscnet"] = history
    del dsc_model, optimizer, scheduler, scaler, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Đối chứng Baseline ResNet-101 (nếu có sẵn hoặc bật TRAIN_BASELINE_IF_MISSING) ---
def _find_baseline_ckpt():
    candidates = [
        ckpt_dir / "resnet101_baseline" / "best.pt",
        DRIVE_BASE / "runs" / BASELINE_RUN_NAME / "checkpoints" / "resnet101" / "best.pt",
        project_root / "checkpoints" / "resnet101" / "best.pt",
    ]
    for cand in candidates:
        if cand.is_file():
            try:
                payload = torch.load(cand, map_location="cpu", weights_only=True)
                if isinstance(payload, dict) and "config" in payload:
                    if payload["config"].get("layer4_stride1") == LAYER4_STRIDE_1:
                        return cand
                elif cand.parent.name != "resnet101":
                    return cand
            except Exception:
                continue
    return None

if COMPARE_WITH_BASELINE:
    base_ckpt = _find_baseline_ckpt()
    if base_ckpt is not None:
        trained_models["resnet101_baseline"] = load_classifier(
            MODEL_NAME, base_ckpt, device=device, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
        )
        try:
            pl = torch.load(base_ckpt, map_location="cpu", weights_only=True)
            all_histories["resnet101_baseline"] = pl.get("history", []) if isinstance(pl, dict) else []
        except Exception:
            all_histories["resnet101_baseline"] = []
        print(f"[Baseline] Nạp ResNet-101 baseline từ {base_ckpt}")
    elif TRAIN_BASELINE_IF_MISSING:
        base_dir = ckpt_dir / "resnet101_baseline"
        base_best = base_dir / "best.pt"
        base_loader = make_loader(
            train_dataset, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, device=device, seed=SEED
        )
        base_model = build_classifier(
            model_name=MODEL_NAME, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
        ).to(device)
        optimizer = build_optimizer(base_model, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
        scheduler = poly_scheduler(optimizer, total_steps=NUM_EPOCHS * len(base_loader), power=POLY_POWER)
        best_map = -1.0
        history_base = []
        for epoch in range(1, NUM_EPOCHS + 1):
            train_loss = train_one_epoch(
                base_model, base_loader, optimizer, criterion, device, scheduler=scheduler,
                use_amp=USE_AMP, desc=f"[Baseline] Ep {epoch:02d}/{NUM_EPOCHS:02d}",
            )
            val_probs, val_targets, val_loss = predict_probs(base_model, val_loader, device, criterion=criterion, leave=False, use_amp=USE_AMP)
            val_cls = compute_classification_metrics(val_targets, val_probs, threshold=CLS_THRESHOLD)
            val_map = val_cls["mAP"]
            history_base.append({
                "backbone": "resnet101_baseline",
                "epoch": epoch,
                "train_loss": round(train_loss, 6),
                "val_loss": round(val_loss, 6),
                "val_mAP": round(val_map, 6),
                "val_macro_f1": round(val_cls["macro_f1"], 6),
            })
            if val_map > best_map:
                best_map = val_map
                torch.save({
                    "backbone": MODEL_NAME,
                    "model_state_dict": base_model.state_dict(),
                    "history": list(history_base),
                    "config": config_record,
                }, base_best)
            print(f"[Baseline] Ep {epoch:02d}/{NUM_EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val mAP: {val_map:.4f}")
        trained_models["resnet101_baseline"] = load_classifier(
            MODEL_NAME, base_best, device=device, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
        )
        all_histories["resnet101_baseline"] = history_base
        del base_model, optimizer, scheduler, base_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("[Baseline] Không tìm thấy checkpoint baseline stride=1 (bỏ qua đối chứng; đặt TRAIN_BASELINE_IF_MISSING=True nếu muốn train).")

# Vẽ đường cong huấn luyện DSCNet
dsc_hist = all_histories.get("resnet101_dscnet", [])
if dsc_hist:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    ep = [r["epoch"] for r in dsc_hist]
    axes[0].plot(ep, [r.get("ce_loss", r["train_loss"]) for r in dsc_hist], marker="o", label="L_CE")
    if "pgcl_loss" in dsc_hist[0]:
        axes[0].plot(ep, [r["pgcl_loss"] for r in dsc_hist], marker="^", label="L_PGCL")
        axes[0].plot(ep, [r["sgcl_loss"] for r in dsc_hist], marker="s", label="L_SGCL")
    axes[0].set_title("DSCNet Loss Components")
    axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

    for v_name, h in all_histories.items():
        if h:
            axes[1].plot([r["epoch"] for r in h], [r["val_mAP"] * 100 for r in h], marker="o", label=v_name)
            axes[2].plot([r["epoch"] for r in h], [r["val_macro_f1"] * 100 for r in h], marker="s", label=v_name)

    axes[1].set_title("Val Classification mAP (%)")
    axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].set_title("Val Macro F1 (%)")
    axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(plots_dir / "dscnet_training_curves.png", dpi=160, bbox_inches="tight")
    plt.show()

## 6. Khởi tạo SAM & Cache Image Embeddings trên trọn bộ **310 ảnh Tune**

Mỗi ảnh trong tập **310 ảnh** Tune chỉ qua SAM Image Encoder đúng **1 lần**, lưu embedding trên CPU và tái sử dụng cho cả **60 trials `CAM + SAM`** lẫn **60 trials `LayerCAM + SAM`**.

In [ ]:
predictor = build_sam_predictor(SAM_MODEL_TYPE, local_sam_ckpt, device=device)
print(f"SamPredictor ({SAM_MODEL_TYPE}) đã sẵn sàng trên {device}")

shared_tune_images = load_eval_samples(
    tune_balanced_dataset,
    image_size=None,
    desc=f"Loading Tune Images & Native GT Masks ({len(tune_balanced_dataset)} ảnh)",
)

if TUNE_SUBSET_SIZE >= len(shared_tune_images):
    tune_samples = list(shared_tune_images)
else:
    tune_step = max(1, len(shared_tune_images) // TUNE_SUBSET_SIZE)
    tune_indices = list(range(0, len(shared_tune_images), tune_step))[:TUNE_SUBSET_SIZE]
    tune_samples = [shared_tune_images[i] for i in tune_indices]

shared_sam_embeds = {}
for local_i, sample in enumerate(tqdm(
    tune_samples, desc=f"Caching Shared SAM ({SAM_MODEL_TYPE}) Embeddings ({len(tune_samples)} ảnh)"
)):
    predictor.set_image(sample["img_u8"])
    shared_sam_embeds[local_i] = get_sam_embedding(predictor)

print(f"Số ảnh Optuna Tune: {len(tune_samples)} | SAM embeddings đã cache: {len(shared_sam_embeds)}")

## 7. Chạy Optuna (`TPESampler`) trên **310 ảnh** cho 4 phương thức (`Vanilla CAM`, `CAM + SAM`, `LayerCAM`, `LayerCAM + SAM`)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_best_params = {}
id_to_seg_idx = {img_id: i for i, img_id in enumerate(seg_train_dataset.image_ids)}
required_optuna_keys = (
    "tune_protocol_version",
    "best_cls_threshold",
    "best_cam_params",
    "best_sam_params",
    "best_layercam_params",
    "best_layercam_sam_params",
)


def _normalize_sam_score_weights(w_cam_iou, w_core_recall, w_sam_iou, w_bg_spill):
    total = max(float(w_cam_iou + w_core_recall + w_sam_iou + w_bg_spill), 1e-8)
    return {
        "w_cam_iou": round(float(w_cam_iou) / total, 5),
        "w_core_recall": round(float(w_core_recall) / total, 5),
        "w_sam_iou": round(float(w_sam_iou) / total, 5),
        "w_bg_spill": round(float(w_bg_spill) / total, 5),
    }


def _sam_params_for_eval(params):
    cam_threshold = float(params["cam_threshold"])
    low_threshold = round(float(params["low_ratio"]) * cam_threshold, 4)
    bg_threshold = round(float(params.get("bg_ratio", 0.0)) * low_threshold, 4) if USE_NEG_POINTS else 0.0
    return {
        "cam_threshold": cam_threshold,
        "low_threshold": low_threshold,
        "bg_threshold": bg_threshold,
        "box_pad": int(params["box_pad"]),
        "cam_pixel_weight": float(params["cam_pixel_weight"]),
        "use_neg_points": USE_NEG_POINTS,
        **_normalize_sam_score_weights(
            w_cam_iou=params["w_cam_iou"],
            w_core_recall=params["w_core_recall"],
            w_sam_iou=params["w_sam_iou"],
            w_bg_spill=params["w_bg_spill"],
        ),
    }


for variant_name, model in trained_models.items():
    v_param_file = optuna_dir / f"{variant_name}_best_params.json"
    if RESUME_IF_EXISTS and v_param_file.is_file():
        loaded_params = json.loads(v_param_file.read_text(encoding="utf-8"))
        if (
            all(k in loaded_params for k in required_optuna_keys)
            and loaded_params.get("sam_model") == SAM_MODEL_TYPE
            and loaded_params.get("tune_protocol_version") == TUNE_PROTOCOL_VERSION
            and loaded_params.get("tune_subset_size") == len(tune_samples)
            and loaded_params.get("layer4_stride1") == LAYER4_STRIDE_1
            and list(loaded_params.get("layercam_layers", [])) == list(LAYERCAM_LAYERS)
            and list(loaded_params.get("cam_scales", [])) == list(CAM_SCALES)
            and loaded_params.get("cam_hflip") == CAM_HFLIP
        ):
            print(f"[Resume] Nạp Optuna bundle '{variant_name}' từ {v_param_file}")
            all_best_params[variant_name] = loaded_params
            continue

    print("\n" + "=" * 80)
    print(f"OPTUNA 4 PHƯƠNG PHÁP ({len(tune_samples)} ẢNH) CHO: {variant_name.upper()}")
    print("=" * 80)

    seg_probs_all, seg_targets_all, _ = predict_probs(
        model, seg_train_loader, device, desc=f"[{variant_name}] Infer Seg/Train Probs", use_amp=USE_AMP
    )

    def obj_cls(trial):
        thr = trial.suggest_float("cls_threshold", 0.25, 0.75, step=0.01)
        return compute_classification_metrics(seg_targets_all, seg_probs_all, threshold=thr)["macro_f1"]

    study_cls = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study_cls.optimize(obj_cls, n_trials=OPTUNA_CLS_TRIALS)
    best_cls_thr = study_cls.best_params["cls_threshold"]
    study_cls.trials_dataframe().to_csv(optuna_dir / f"{variant_name}_trials_cls.csv", index=False)
    print(f"[{variant_name}] Best cls_threshold = {best_cls_thr:.2f} | Macro-F1 = {study_cls.best_value:.4f}")

    tune_probs = np.stack([seg_probs_all[id_to_seg_idx[item["img_id"]]] for item in tune_samples])
    is_dsc = isinstance(model, DSCNetClassifier)
    if is_dsc:
        cam_fns = {
            "cam": make_dscnet_cam_fn(model, method="cam", scales=CAM_SCALES, hflip=CAM_HFLIP, use_refined=True),
            "layercam": make_dscnet_cam_fn(
                model, method="layercam", scales=CAM_SCALES, hflip=CAM_HFLIP,
                layers=LAYERCAM_LAYERS, shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
            ),
        }
    else:
        cam_fns = {
            "cam": make_cam_fn(model, method="cam", scales=CAM_SCALES, hflip=CAM_HFLIP),
            "layercam": make_cam_fn(
                model, method="layercam", scales=CAM_SCALES, hflip=CAM_HFLIP,
                layers=LAYERCAM_LAYERS, shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
            ),
        }

    cam_caches = cache_cams(
        tune_samples,
        tune_probs,
        cls_threshold=best_cls_thr,
        cam_fns=cam_fns,
        desc=f"[{variant_name}] Caching Multi-Scale CAM & LayerCAM ({len(tune_samples)} ảnh)",
    )

    def _run_cam_study(cache_list, n_trials, csv_name, label):
        def _obj(trial):
            cam_thr = trial.suggest_float("cam_threshold", 0.03, 0.45, step=0.01)
            evaluator = SegmentationEvaluator(num_classes=21, ignore_index=255)
            for base_item, cam_item in zip(tune_samples, cache_list):
                pseudo_mask = gen_cam_pseudo_mask(
                    cam_item["cams"], cam_item["active_classes"], cam_threshold=cam_thr
                )
                evaluator.update(pseudo_mask, base_item["gt_mask"])
            return evaluator.compute()["mIoU"]

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_obj, n_trials=n_trials, show_progress_bar=True)
        study.trials_dataframe().to_csv(optuna_dir / csv_name, index=False)
        print(f"[{variant_name} | {label}] Best: {study.best_params} | Tune mIoU = {study.best_value:.4f}")
        return study

    def _run_sam_study(cache_list, n_trials, csv_name, label):
        def _obj(trial):
            cam_threshold = trial.suggest_float("cam_threshold", 0.04, 0.38, step=0.02)
            low_ratio = trial.suggest_float("low_ratio", 0.25, 0.90, step=0.05)
            low_threshold = round(low_ratio * cam_threshold, 4)
            bg_ratio = trial.suggest_float("bg_ratio", 0.05, 0.60, step=0.05) if USE_NEG_POINTS else 0.0
            bg_threshold = round(bg_ratio * low_threshold, 4)
            box_pad = trial.suggest_int("box_pad", 4, 64, step=4)
            w_cam_iou = trial.suggest_float("w_cam_iou", 0.20, 0.70, step=0.05)
            w_core_recall = trial.suggest_float("w_core_recall", 0.05, 0.55, step=0.05)
            w_sam_iou = trial.suggest_float("w_sam_iou", 0.10, 0.40, step=0.05)
            w_bg_spill = trial.suggest_float("w_bg_spill", 0.20, 1.00, step=0.05)
            cam_pixel_weight = trial.suggest_float("cam_pixel_weight", 0.35, 0.90, step=0.05)
            score_weights = _normalize_sam_score_weights(
                w_cam_iou=w_cam_iou, w_core_recall=w_core_recall,
                w_sam_iou=w_sam_iou, w_bg_spill=w_bg_spill,
            )

            evaluator = SegmentationEvaluator(num_classes=21, ignore_index=255)
            for local_i, base_item in enumerate(tune_samples):
                cam_item = cache_list[local_i]
                set_sam_embedding(predictor, shared_sam_embeds[local_i], device=device)
                pseudo_mask = gen_sam_pseudo_mask(
                    predictor=predictor,
                    image_rgb=base_item["img_rgb"],
                    cams=cam_item["cams"],
                    class_indices=cam_item["active_classes"],
                    cam_threshold=cam_threshold,
                    low_threshold=low_threshold,
                    bg_threshold=bg_threshold,
                    use_neg_points=USE_NEG_POINTS,
                    box_pad=box_pad,
                    encode_image=False,
                    w_cam_iou=score_weights["w_cam_iou"],
                    w_core_recall=score_weights["w_core_recall"],
                    w_sam_iou=score_weights["w_sam_iou"],
                    w_bg_spill=score_weights["w_bg_spill"],
                    cam_pixel_weight=cam_pixel_weight,
                )
                evaluator.update(pseudo_mask, base_item["gt_mask"])
            return evaluator.compute()["mIoU"]

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_obj, n_trials=n_trials, show_progress_bar=True)
        study.trials_dataframe().to_csv(optuna_dir / csv_name, index=False)
        print(f"[{variant_name} | {label}] Best: {study.best_params} | Tune mIoU = {study.best_value:.4f}")
        return study

    study_cam = _run_cam_study(
        cam_caches["cam"], OPTUNA_CAM_TRIALS, f"{variant_name}_trials_cam.csv", "Vanilla CAM"
    )
    study_sam = _run_sam_study(
        cam_caches["cam"], OPTUNA_SAM_TRIALS, f"{variant_name}_trials_sam.csv", f"CAM + SAM ({SAM_MODEL_TYPE})"
    )
    study_layercam = _run_cam_study(
        cam_caches["layercam"], OPTUNA_LAYERCAM_TRIALS, f"{variant_name}_trials_layercam.csv", "LayerCAM"
    )
    study_layercam_sam = _run_sam_study(
        cam_caches["layercam"], OPTUNA_LAYERCAM_SAM_TRIALS,
        f"{variant_name}_trials_layercam_sam.csv", f"LayerCAM + SAM ({SAM_MODEL_TYPE})",
    )

    bundle = {
        "variant": variant_name,
        "backbone": MODEL_NAME,
        "image_size": IMAGE_SIZE,
        "tune_protocol_version": TUNE_PROTOCOL_VERSION,
        "tune_subset_size": len(tune_samples),
        "eval_protocol": "original_resolution",
        "use_neg_points": USE_NEG_POINTS,
        "layer4_stride1": LAYER4_STRIDE_1,
        "sam_model": SAM_MODEL_TYPE,
        "cam_scales": list(CAM_SCALES),
        "cam_hflip": CAM_HFLIP,
        "layercam_layers": list(LAYERCAM_LAYERS),
        "layercam_shallow_gamma": LAYERCAM_SHALLOW_GAMMA,
        "best_cls_threshold": best_cls_thr,
        "best_cam_params": study_cam.best_params,
        "best_sam_params": _sam_params_for_eval(study_sam.best_params),
        "best_layercam_params": study_layercam.best_params,
        "best_layercam_sam_params": _sam_params_for_eval(study_layercam_sam.best_params),
        "tune_best_miou": {
            "vanilla_cam": round(float(study_cam.best_value), 4),
            "cam_sam": round(float(study_sam.best_value), 4),
            "layercam": round(float(study_layercam.best_value), 4),
            "layercam_sam": round(float(study_layercam_sam.best_value), 4),
        },
    }
    v_param_file.write_text(json.dumps(bundle, indent=2, ensure_ascii=False), encoding="utf-8")
    all_best_params[variant_name] = bundle

    del cam_caches
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(optuna_dir / "all_variants_best_params.json").write_text(
    json.dumps(all_best_params, indent=2, ensure_ascii=False), encoding="utf-8"
)

shared_sam_embeds.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Biểu đồ hội tụ Optuna ---
fig, axes = plt.subplots(2, 3, figsize=(18, 9.2))
colors = {"resnet101_dscnet": "#2ca02c", "resnet101_baseline": "#1f77b4"}
specs = [
    (axes[0, 0], "trials_cam.csv", True, "Vanilla CAM: threshold vs mIoU"),
    (axes[0, 1], "trials_sam.csv", False, f"CAM + SAM ({SAM_MODEL_TYPE}) convergence"),
    (axes[1, 0], "trials_layercam.csv", True, "LayerCAM: threshold vs mIoU"),
    (axes[1, 1], "trials_layercam_sam.csv", False, f"LayerCAM + SAM ({SAM_MODEL_TYPE}) convergence"),
]
for ax, csv_suffix, is_threshold, title in specs:
    for v_name in trained_models:
        csv_path = optuna_dir / f"{v_name}_{csv_suffix}"
        if not csv_path.is_file():
            continue
        df_t = pd.read_csv(csv_path)
        c = colors.get(v_name)
        if is_threshold:
            df_t = df_t.sort_values("params_cam_threshold")
            ax.plot(df_t["params_cam_threshold"], df_t["value"] * 100, marker="o", color=c, label=v_name)
        else:
            ax.plot(df_t["number"], df_t["value"].cummax() * 100, marker=".", linewidth=2, color=c, label=v_name)
    ax.set_title(title)
    ax.set_xlabel("cam_threshold" if is_threshold else "Trial")
    ax.set_ylabel("Tune mIoU (%)")
    ax.grid(alpha=0.3); ax.legend()

tune_keys = ["vanilla_cam", "cam_sam", "layercam", "layercam_sam"]
tune_labels = ["CAM", "CAM+SAM", "LayerCAM", "LCAM+SAM"]
v_list = list(trained_models.keys())
for idx_v, v_name in enumerate(v_list):
    vals = [all_best_params[v_name]["tune_best_miou"][k] * 100 for k in tune_keys]
    pos = np.arange(len(tune_keys)) + (idx_v - (len(v_list) - 1) / 2) * 0.30
    axes[0, 2].bar(pos, vals, width=0.30, color=colors.get(v_name), label=v_name)
axes[0, 2].set_xticks(np.arange(len(tune_keys)))
axes[0, 2].set_xticklabels(tune_labels, rotation=15)
axes[0, 2].set_title(f"Best Tune mIoU (%) trên {len(tune_samples)} ảnh")
axes[0, 2].set_ylabel("Tune mIoU (%)")
axes[0, 2].grid(axis="y", alpha=0.3); axes[0, 2].legend()

axes[1, 2].axis("off")
axes[1, 2].text(
    0.0, 0.9,
    f"Ghi chú:\n"
    f"- Paper: DSCNet (arXiv:2405.04913)\n"
    f"- Train: {len(df_train):,} ảnh (Main/trainval \\ Seg/val)\n"
    f"- Optuna Tune: {len(tune_samples)} ảnh cân bằng (0% overlap)\n"
    f"- Eval cuối: {len(df_val):,} ảnh Seg/val (native res)\n"
    f"- SAM: {SAM_MODEL_TYPE}",
    fontsize=11, va="top",
)
plt.tight_layout()
plt.savefig(plots_dir / "optuna_history.png", dpi=160)
plt.show()

# --- Hyperparameter Importance (2 nhánh SAM) ---
from sklearn.ensemble import RandomForestRegressor

def compute_csv_param_importance(csv_path: Path, seed: int = 42) -> pd.Series:
    df = pd.read_csv(csv_path).dropna(subset=["value"])
    param_cols = [c for c in df.columns if c.startswith("params_")]
    if len(df) < 4 or not param_cols:
        return pd.Series(dtype=float)
    X = df[param_cols].copy()
    y = df["value"].values
    valid_cols = [c for c in param_cols if X[c].nunique() > 1]
    if not valid_cols:
        return pd.Series(1.0 / len(param_cols), index=[c.replace("params_", "") for c in param_cols])
    rf = RandomForestRegressor(n_estimators=128, max_depth=6, random_state=seed)
    rf.fit(X[valid_cols], y)
    imp = pd.Series(rf.feature_importances_, index=[c.replace("params_", "") for c in valid_cols])
    return (imp / imp.sum()) if imp.sum() > 0 else imp

fig_imp, axes_imp = plt.subplots(2, len(v_list), figsize=(6.2 * len(v_list), 8.4), squeeze=False)
importance_records = []
method_imp_specs = [
    ("CAM + SAM", "trials_sam.csv", "#e6550d"),
    ("LayerCAM + SAM", "trials_layercam_sam.csv", "#c44e52"),
]
for row_idx, (m_title, csv_suffix, bar_color) in enumerate(method_imp_specs):
    for col_idx, v_name in enumerate(v_list):
        ax = axes_imp[row_idx, col_idx]
        csv_file = optuna_dir / f"{v_name}_{csv_suffix}"
        if csv_file.is_file():
            imp_series = compute_csv_param_importance(csv_file, seed=SEED).sort_values(ascending=True)
            bars = ax.barh(imp_series.index, imp_series.values * 100, color=bar_color, alpha=0.88, edgecolor="black", height=0.6)
            for bar in bars:
                w_val = bar.get_width()
                ax.text(w_val + 1.0, bar.get_y() + bar.get_height() / 2, f"{w_val:.1f}%", va="center", fontsize=8.5, fontweight="bold")
            ax.set_xlim(0, max(100.0, float(imp_series.max() * 100) + 18.0))
            for p_name, p_val in imp_series.sort_values(ascending=False).items():
                importance_records.append({
                    "Method": m_title, "Variant": v_name, "Hyperparameter": p_name,
                    "Importance (%)": round(float(p_val * 100), 2),
                })
        ax.set_title(f"{m_title} — {v_name}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Mức độ ảnh hưởng (%)"); ax.grid(axis="x", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig(plots_dir / "optuna_param_importance.png", dpi=160)
plt.show()
if importance_records:
    df_imp = pd.DataFrame(importance_records).sort_values(["Method", "Variant", "Importance (%)"], ascending=[True, True, False])
    df_imp.to_csv(optuna_dir / "param_importance_summary.csv", index=False)
    display(df_imp)

## 8. Đánh giá cuối trên `Segmentation/val.txt` (`1,449 ảnh`, độ phân giải gốc)

Đánh giá đầy đủ cả **4 phương thức** (`Vanilla CAM`, `CAM + SAM`, `LayerCAM`, `LayerCAM + SAM`) với **Shared SAM Image Encoding** (`predictor.set_image` chạy đúng 1 lần cho mỗi ảnh Val).

In [ ]:
val_probs_by_variant = {}
cls_summary_rows = []
val_targets_all = None

for variant_name, model in trained_models.items():
    cls_thr = all_best_params[variant_name]["best_cls_threshold"]
    probs_arr, targets_arr, _ = predict_probs(
        model, val_loader, device, desc=f"Val Classification (1,449 ảnh) [{variant_name}]", use_amp=USE_AMP
    )
    if val_targets_all is None:
        val_targets_all = targets_arr
    val_probs_by_variant[variant_name] = probs_arr

    cls_metrics = compute_classification_metrics(val_targets_all, probs_arr, threshold=cls_thr)
    log_eval_results(
        experiment_name=f"{variant_name}_Classification",
        metrics=cls_metrics,
        config={**config_record, "variant": variant_name, "cls_threshold": cls_thr},
        log_dir=run_dir,
    )
    cls_summary_rows.append(
        cls_summary_row(cls_metrics, percent=True, Variant=variant_name, **{"Best Cls Thr": cls_thr})
    )

cls_summary_df = pd.DataFrame(cls_summary_rows)
cls_summary_df.to_csv(run_dir / "val_classification_summary.csv", index=False)
print("=== KẾT QUẢ PHÂN LOẠI ĐA NHÃN TRÊN VAL (1,449 ẢNH) ===")
display(cls_summary_df)

pipelines = {}
for variant_name, model in trained_models.items():
    if isinstance(model, DSCNetClassifier):
        pipelines[variant_name] = build_dscnet_optuna_pipeline(
            model=model,
            probs=val_probs_by_variant[variant_name],
            best_params=all_best_params[variant_name],
            predictor=predictor,
            encode_image=False,
            scales=CAM_SCALES,
            hflip=CAM_HFLIP,
            layercam_layers=LAYERCAM_LAYERS,
            shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
            use_refined_cam=True,
        )
    else:
        pipelines[variant_name] = build_optuna_pipeline(
            model=model,
            probs=val_probs_by_variant[variant_name],
            best_params=all_best_params[variant_name],
            predictor=predictor,
            encode_image=False,
            scales=CAM_SCALES,
            hflip=CAM_HFLIP,
            layercam_layers=LAYERCAM_LAYERS,
            shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
        )

computed_seg_metrics = evaluate_segmentation(
    val_dataset,
    pipelines,
    image_size=None,
    prepare_fn=predictor.set_image,
    desc=f"Val Segmentation (1,449 ảnh — {len(pipelines)} variant × 4 methods, shared SAM {SAM_MODEL_TYPE})",
)

seg_summary_rows = []
per_class_records = {"Class": ["background", *CLASSES]}
for variant_name in trained_models:
    for method_label in METHODS:
        metrics = computed_seg_metrics[variant_name][method_label]
        safe_method = method_label.replace(" ", "").replace("+", "_")
        log_eval_results(
            experiment_name=f"{variant_name}_{safe_method}",
            metrics=metrics,
            config={**all_best_params[variant_name], "method": method_label},
            log_dir=run_dir,
        )
        seg_summary_rows.append(
            seg_summary_row(metrics, percent=True, Variant=variant_name, Method=method_label)
        )
        per_class_records[f"{variant_name} | {method_label}"] = [
            round(v * 100, 2) for v in metrics["iou_per_class"]
        ]

seg_df = pd.DataFrame(seg_summary_rows)
seg_df.to_csv(run_dir / "val_segmentation_summary.csv", index=False)
pd.DataFrame(per_class_records).to_csv(run_dir / "val_per_class_iou.csv", index=False)

print("\n=== BẢNG TỔNG HỢP PHÂN ĐOẠN TRÊN VAL (1,449 ẢNH) ===")
display(seg_df)

variants_list = list(trained_models.keys())
fig, ax = plt.subplots(figsize=(11.5, 5.4))
x = np.arange(len(variants_list))
width = 0.2
method_colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]
for idx, method_label in enumerate(METHODS):
    values = [computed_seg_metrics[v][method_label]["mIoU"] * 100 for v in variants_list]
    bars = ax.bar(
        x + (idx - (len(METHODS) - 1) / 2) * width,
        values, width=width, label=method_label, color=method_colors[idx],
    )
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5, f"{h:.1f}%", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([v.upper() for v in variants_list], fontsize=11, fontweight="bold")
ax.set_ylabel("Val mIoU (%)")
ax.set_title(f"DSCNet ResNet-101 (arXiv:2405.04913) — Val mIoU 4 phương pháp (SAM {SAM_MODEL_TYPE})")
ax.set_ylim(0, seg_df["mIoU (%)"].max() + 8)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.10))
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(plots_dir / "val_miou_dscnet.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Trực quan hoá định tính & Tổng kết kết quả

In [ ]:
sample_indices = [0, 12, 25, 42]
rows = []
for idx in sample_indices:
    sample = load_eval_sample(val_dataset, idx, image_size=None)
    predictor.set_image(sample["img_u8"])

    dsc_pipe = pipelines["resnet101_dscnet"]
    dsc_out = predict_sample_masks(sample, dsc_pipe, dsc_pipe.probs[idx])
    active_names = [CLASSES[c] for c in dsc_out["active_classes"]]

    row = [
        image_cell(sample["img_rgb"], f"{sample['img_id']}\n{active_names}"),
        mask_cell(sample["gt_mask"], "Ground Truth"),
        mask_cell(dsc_out["masks"]["Vanilla CAM"], "DSCNet CAM"),
        mask_cell(dsc_out["masks"]["CAM + SAM"], "DSCNet CAM+SAM"),
        mask_cell(dsc_out["masks"]["LayerCAM"], "DSCNet LayerCAM"),
        mask_cell(dsc_out["masks"]["LayerCAM + SAM"], "DSCNet LCAM+SAM"),
    ]
    rows.append(row)

plot_grid(rows, cell_size=3.0, save_path=plots_dir / "val_qualitative_dscnet.png")

print("=" * 80)
print(f"HOÀN TẤT DSCNet (arXiv:2405.04913) — kết quả lưu tại: {run_dir}")
print("=" * 80)
if "seg_df" in globals() and len(seg_df) > 0:
    best_row = seg_df.sort_values("mIoU (%)", ascending=False).iloc[0]
    print(f"Best mIoU: {best_row['mIoU (%)']:.2f}% | {best_row['Variant']} | {best_row['Method']}")

for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        print(f" - {path.relative_to(run_dir)} ({path.stat().st_size / 1024:,.1f} KB)")